# Audio Fundamentals — Analyse d’un fichier audio

Ce notebook applique les notions fondamentales du traitement audio :

- chargement d’un fichier WAV ;
- fréquence d’échantillonnage ;
- nombre d’échantillons ;
- durée ;
- audio mono et stéréo ;
- waveform ;
- amplitude et clipping ;
- rééchantillonnage ;
- comparaison avant et après rééchantillonnage ;
- analyse fréquentielle avec la FFT.

L’objectif est de comprendre comment un signal audio réel est transformé en données numériques utilisables par un modèle d’intelligence artificielle.

## 1. Importation des bibliothèques

Nous importons les bibliothèques nécessaires à l’analyse :

- **NumPy** pour manipuler les tableaux numériques ;
- **Matplotlib** pour afficher les graphiques ;
- **SoundFile** pour lire et enregistrer des fichiers WAV ;
- **Librosa** pour effectuer des traitements audio comme le rééchantillonnage ;
- **IPython.display** pour écouter les fichiers directement dans Jupyter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
import librosa.display

## 2. Chargement du fichier audio

Un fichier audio numérique contient une suite de valeurs représentant l’amplitude du signal à différents instants.

La fonction sf.read() retourne deux informations :

1. `audio` : le tableau contenant les échantillons ;
2. `sample_rate` : le nombre d’échantillons enregistrés chaque seconde.

Par exemple, un sample rate de 44 100 Hz signifie que le signal contient 44 100 mesures par seconde.


In [ ]:
import librosa

audio, sample_rate = librosa.load(
    "voice.mp3",
    sr=None,
    mono=False
)

print("Sample rate :", sample_rate)
print("Shape :", audio.shape)

In [ ]:
import soundfile as sf

audio, sample_rate = sf.read("voice_excerpt.wav")

print("Sample rate :", sample_rate)
print("Shape :", audio.shape)
print("Type de données :", audio.dtype)

## 3. Vérification de la forme du signal

La propriété `shape` indique la structure du tableau audio.

Un résultat comme :

```text
(882000,)

In [ ]:
if audio.ndim == 2:
    audio = audio.mean(axis=1)
    print("Audio converti en mono")

print("Nouvelle shape :", audio.shape)

## 4. Calcul de la durée

La durée d’un audio est obtenue en divisant son nombre d’échantillons par sa fréquence d’échantillonnage.

La formule est :

\[
\text{durée} =
\frac{\text{nombre d'échantillons}}
{\text{sample rate}}
\]

Par exemple, un signal contenant 882 000 échantillons à 44 100 Hz dure :

\[
\frac{882000}{44100} = 20\ \text{secondes}
\]

In [ ]:
duration = len(audio) / sample_rate

print(f"Durée : {duration:.2f} secondes")

## 5. Conversion en mono

De nombreux modèles de reconnaissance vocale et de traitement de la parole attendent un signal mono.

Lorsqu’un audio est stéréo, il contient deux valeurs pour chaque instant :

- une pour le canal gauche ;
- une pour le canal droit.

Pour obtenir un signal mono simple, nous calculons ici la moyenne des deux canaux.

Cette conversion réduit le tableau de :

```text
(nombre d'échantillons, 2)

In [ ]:
if audio.ndim == 2:
    print("Audio stéréo détecté :", audio.shape)

    # Moyenne du canal gauche et du canal droit
    audio = audio.mean(axis=1)

    print("Audio converti en mono.")
else:
    print("L'audio est déjà mono.")

print("Nouvelle shape :", audio.shape)

## 6. Visualisation de la waveform

La waveform représente l’amplitude du signal audio en fonction du temps.

- l’axe horizontal représente le temps en secondes ;
- l’axe vertical représente l’amplitude ;
- une zone presque plate correspond généralement à un silence ;
- une zone dense ou avec de grands pics correspond à une activité sonore.

Chaque point du graphique correspond à un échantillon du fichier audio.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

time = np.arange(len(audio)) / sample_rate

plt.figure(figsize=(14, 4))
plt.plot(time, audio)
plt.xlabel("Temps en secondes")
plt.ylabel("Amplitude")
plt.title("Waveform de l'audio")
plt.show()

## 7. Vérification du niveau d’amplitude

Les fichiers audio chargés sous forme de nombres décimaux utilisent généralement des valeurs comprises entre `-1` et `1`.

Nous vérifions :

- l’amplitude minimale ;
- l’amplitude maximale ;
- le pic absolu maximal.

Un pic proche de `1.0` signifie que le signal approche sa limite maximale.

Lorsqu’un signal dépasse la plage autorisée, il peut être coupé. Ce phénomène est appelé **clipping** et peut provoquer une distorsion audible.

In [ ]:
print("Amplitude minimale :", audio.min())
print("Amplitude maximale :", audio.max())
print("Pic absolu :", np.abs(audio).max())

## 8. Recherche d’un clipping potentiel

Le clipping apparaît lorsque l’amplitude d’un signal atteint ou dépasse la limite du format audio.

Nous comptons ici le nombre d’échantillons dont l’amplitude absolue est supérieure ou égale à `0.999`.

La présence de quelques valeurs proches de la limite ne prouve pas toujours une distorsion importante. En revanche, un grand nombre d’échantillons bloqués à la limite peut indiquer un signal saturé.

In [ ]:
clipped_samples = np.sum(np.abs(audio) >= 0.999)

print("Échantillons potentiellement clippés :", clipped_samples)
print(
    "Pourcentage potentiel :",
    f"{100 * clipped_samples / len(audio):.6f} %"
)

## 9. Rééchantillonnage à 16 kHz

Le sample rate indique le nombre d’échantillons présents dans une seconde d’audio.

De nombreux modèles de reconnaissance vocale utilisent un sample rate de 16 000 Hz.

Nous convertissons donc le signal original vers 16 kHz avec un véritable algorithme de rééchantillonnage.

Le rééchantillonnage doit :

- modifier le nombre d’échantillons ;
- conserver approximativement la même durée ;
- appliquer un filtrage anti-aliasing ;
- éviter de modifier la vitesse de lecture.

Il ne faut pas simplement supprimer manuellement certains échantillons.

## 10. Vérification de la durée après rééchantillonnage

Le rééchantillonnage change le nombre d’échantillons, mais il ne doit pas changer la durée réelle de l’audio.

La version à 16 kHz contient moins de mesures par seconde que la version originale.

Cependant, lorsqu’elle est lue avec le bon sample rate, elle doit rester aussi longue et conserver la même vitesse.

In [ ]:
import librosa

target_sample_rate = 16000

audio_16k = librosa.resample(
    audio,
    orig_sr=sample_rate,
    target_sr=target_sample_rate
)

print("Sample rate original :", sample_rate)
print("Sample rate cible :", target_sample_rate)
print("Échantillons avant :", len(audio))
print("Échantillons après :", len(audio_16k))
print("Durée avant :", len(audio) / sample_rate)
print("Durée après :", len(audio_16k) / target_sample_rate)

## 11. Sauvegarde de la version à 16 kHz

Nous enregistrons le signal rééchantillonné dans un nouveau fichier WAV.

Le fichier original est conservé. Cela permet de comparer :

- la taille des deux fichiers ;
- leur qualité sonore ;
- leur nombre d’échantillons ;
- leur contenu fréquentiel.

Il est important de préciser le sample rate correct lors de la sauvegarde.

In [ ]:
sf.write(
    "voice_excerpt_16k.wav",
    audio_16k,
    target_sample_rate
)

print("Fichier sauvegardé")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

time_original = np.arange(len(audio)) / sample_rate
time_16k = np.arange(len(audio_16k)) / target_sample_rate

## 12. Affichage de la waveform à 16 kHz

Nous affichons maintenant la waveform de la version rééchantillonnée.

Sa forme générale doit rester proche de celle du signal original.

La principale différence est que le signal à 16 kHz contient moins de points, car il ne conserve que 16 000 échantillons par seconde.

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(time_original, audio)
plt.xlabel("Temps en secondes")
plt.ylabel("Amplitude")
plt.title(f"Waveform originale — {sample_rate} Hz")
plt.show()

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(time_16k, audio_16k)
plt.xlabel("Temps en secondes")
plt.ylabel("Amplitude")
plt.title(f"Waveform resamplée — {target_sample_rate} Hz")
plt.show()

## 13. Comparaison détaillée sur 20 millisecondes

Sur l’ensemble du fichier, les waveforms sont trop denses pour observer chaque échantillon.

Nous sélectionnons donc une fenêtre de 20 millisecondes.

Sur cette durée :

- un signal à 44 100 Hz possède environ 882 échantillons ;
- un signal à 16 000 Hz possède environ 320 échantillons.

Cette comparaison permet de visualiser directement la différence de résolution temporelle.

In [ ]:
start_time = 2.0
end_time = 2.02

In [ ]:
start_original = int(start_time * sample_rate)
end_original = int(end_time * sample_rate)

plt.figure(figsize=(14, 4))
plt.plot(
    time_original[start_original:end_original],
    audio[start_original:end_original],
    marker="."
)
plt.xlabel("Temps en secondes")
plt.ylabel("Amplitude")
plt.title("Zoom waveform originale — 20 ms")
plt.show()

## 14. Zoom sur la version rééchantillonnée

Nous affichons exactement la même période temporelle dans la version à 16 kHz.

Le signal conserve une forme similaire, mais il est représenté avec moins de points.

Cela illustre le principe du rééchantillonnage : diminuer la quantité de données tout en conservant les informations utiles pour l’application ciblée.

In [ ]:
start_16k = int(start_time * target_sample_rate)
end_16k = int(end_time * target_sample_rate)

plt.figure(figsize=(14, 4))
plt.plot(
    time_16k[start_16k:end_16k],
    audio_16k[start_16k:end_16k],
    marker="."
)
plt.xlabel("Temps en secondes")
plt.ylabel("Amplitude")
plt.title("Zoom waveform 16 kHz — 20 ms")
plt.show()

## 15. Comparaison auditive

Nous écoutons maintenant les versions originale et rééchantillonnée.

La vitesse et la durée doivent rester identiques.

Une fréquence d’échantillonnage plus faible peut légèrement réduire la précision des fréquences les plus aiguës. Pour la parole, 16 kHz reste néanmoins suffisant pour de nombreux modèles de reconnaissance vocale.

Si la voix est accélérée ou ralentie, cela indique généralement que le signal est lu avec un mauvais sample rate.

In [ ]:
from IPython.display import Audio, display

In [ ]:
display(Audio(audio, rate=sample_rate))

In [ ]:
display(Audio(audio_16k, rate=target_sample_rate))

## 16. Analyse fréquentielle avec la FFT

La waveform représente l’amplitude du signal en fonction du temps.

La transformée de Fourier permet d’analyser le même signal dans le domaine fréquentiel.

Elle répond à la question :

> Quelles fréquences sont présentes dans le signal, et avec quelle intensité ?

La **FFT**, ou Fast Fourier Transform, est un algorithme efficace permettant de calculer cette transformation.

Pour simplifier l’analyse, nous utilisons uniquement un segment de quelques secondes.

In [ ]:
segment_duration = 3
segment = audio_16k[:segment_duration * target_sample_rate]

## 17. Calcul du spectre fréquentiel

La fonction `np.fft.rfft()` calcule la FFT d’un signal réel.

La fonction `np.fft.rfftfreq()` associe chaque valeur obtenue à une fréquence exprimée en hertz.

Nous calculons ensuite la magnitude avec `np.abs()` afin de mesurer l’importance de chaque fréquence dans le signal.

In [ ]:
fft_values = np.fft.rfft(segment)
fft_frequencies = np.fft.rfftfreq(
    len(segment),
    d=1 / target_sample_rate
)

magnitudes = np.abs(fft_values)

## 18. Visualisation du spectre fréquentiel

Le graphique suivant affiche :

- les fréquences en hertz sur l’axe horizontal ;
- leur magnitude sur l’axe vertical.

Comme le signal est échantillonné à 16 kHz, la fréquence maximale représentable est de 8 kHz.

Cette limite correspond à la fréquence de Nyquist :

\[
f_{\text{Nyquist}} =
\frac{f_{\text{échantillonnage}}}{2}
\]

Ainsi :

\[
\frac{16000}{2} = 8000\ \text{Hz}
\]

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(fft_frequencies, magnitudes)
plt.xlabel("Fréquence en Hz")
plt.ylabel("Magnitude")
plt.title("Spectre fréquentiel — FFT")
plt.xlim(0, 8000)
plt.show()

## 19. Affichage du spectre en décibels

Les magnitudes du signal peuvent avoir des valeurs très différentes.

Quelques fréquences dominantes peuvent rendre les autres presque invisibles sur un graphique linéaire.

Nous convertissons donc les magnitudes en décibels avec une échelle logarithmique :

\[
\text{magnitude en dB}
=
20 \log_{10}(\text{magnitude})
\]

Une très petite valeur est ajoutée pour éviter de calculer le logarithme de zéro.

In [ ]:
magnitudes_db = 20 * np.log10(magnitudes + 1e-10)

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(fft_frequencies, magnitudes_db)
plt.xlabel("Fréquence en Hz")
plt.ylabel("Magnitude en dB")
plt.title("Spectre fréquentiel en décibels")
plt.xlim(0, 8000)
plt.show()

## 20. Limite d’une FFT calculée sur tout un segment

La FFT indique quelles fréquences sont présentes dans l’ensemble du segment analysé.

Cependant, elle ne précise pas à quel moment chaque fréquence apparaît.

Par exemple, elle peut détecter une fréquence aiguë présente pendant trois secondes, mais elle ne permet pas de savoir si cette fréquence apparaît au début, au milieu ou à la fin.

Pour conserver simultanément :

- l’information temporelle ;
- l’information fréquentielle ;

on utilise la **Short-Time Fourier Transform**, ou STFT.

La STFT découpe l’audio en petites fenêtres et calcule une FFT pour chacune. Le résultat peut ensuite être représenté sous forme de spectrogramme.

# Conclusion

Dans ce notebook, nous avons appris à :

- charger un fichier audio numérique ;
- identifier son sample rate ;
- comprendre la forme de son tableau ;
- calculer sa durée ;
- distinguer un signal mono d’un signal stéréo ;
- convertir un audio stéréo en mono ;
- visualiser une waveform ;
- vérifier les amplitudes et le clipping ;
- rééchantillonner proprement un audio à 16 kHz ;
- comparer les versions originale et rééchantillonnée ;
- écouter les deux signaux ;
- passer du domaine temporel au domaine fréquentiel avec la FFT ;
- comprendre la fréquence de Nyquist ;
- identifier les limites d’une FFT globale.

La prochaine étape sera d’étudier la STFT, les spectrogrammes, l’échelle Mel et les log-mel spectrogrammes dans un nouveau notebook :

```text
spectrograms_and_mel_features.ipynb